# 🔥 ITEC Fine-tune — ปรับ backbone แล้วพิสูจน์ว่าคุ้มไหม

> **โปรเจกต์นี้แยกจาก `scripts/itec/category/`** — ที่นั่นคืองานจัดหมวด ที่นี่คืองานปรับ backbone
> เอกสาร → Obsidian `ITEC Model - Fine-tune Design`

---

## ต่างจากโน้ตบุ๊กหลักตรงไหน

```
โน้ตบุ๊กหลัก   ItemName → [ backbone ] → เวกเตอร์ → [ LinearSVC ] → หมวด
                            ❄️ แช่แข็ง               🔥 เรียนตรงนี้

ที่นี่          ItemName → [ backbone ] → เวกเตอร์ → [ LinearSVC ] → หมวด
                            🔥 ปรับด้วย              🔥 เรียนใหม่
```

**เปลี่ยนแค่ชั้นแรก** classifier ยังเป็นตัวเดิมพารามิเตอร์เดิม — จะได้ชี้ได้ว่าอะไรทำให้ดีขึ้น

## ทำ 4 ขั้นในรอบเดียว

```
① แบ่ง train/test       seed 42 · stratify
② วัด baseline          backbone แช่แข็ง → LinearSVC → test
③ fine-tune             เฉพาะ train เท่านั้น ห้ามให้เห็น test
④ วัดใหม่                backbone ที่ปรับ → LinearSVC ตัวเดิม → test ชุดเดิม
```

## ⚠️ อ่านก่อนรัน

**label มาจากกฎ keyword ที่เราเขียนเอง ไม่ใช่ความจริง**

```
fine-tune = สอนให้ embedding แยกตาม label ที่ให้
label ผิด  = สอนให้แยกผิดแม่นขึ้น
```

**ผลอาจออกมาว่า "ไม่ช่วย" หรือ "แย่ลง" ซึ่งเป็นคำตอบที่มีค่า** — พิสูจน์ว่าคอขวดไม่ได้อยู่ที่ embedding


## 1 · Setup — รันได้ทั้ง local · Colab · Kaggle

In [ ]:
import sys, os, subprocess, gzip, shutil, zipfile
from pathlib import Path

IN_COLAB  = "google.colab" in sys.modules
IN_KAGGLE = Path("/kaggle/input").exists()
ENV = "colab" if IN_COLAB else "kaggle" if IN_KAGGLE else "local"

for pkg, mod in [("sentence-transformers", "sentence_transformers"),
                 ("datasets", "datasets"), ("accelerate", "accelerate")]:
    try:
        __import__(mod)
    except ImportError:
        print(f"ติดตั้ง {pkg} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)


def _ungz(src, dst):
    with gzip.open(src, "rb") as f, open(dst, "wb") as o:
        shutil.copyfileobj(f, o)


if ENV == "local":
    SCRIPTS = Path.cwd()
elif ENV == "colab":
    SCRIPTS = Path("/content"); os.chdir(SCRIPTS)
    if not Path("dim_item_itec.csv").exists():
        print("เลือกไฟล์  dim_item_itec.csv.gz")
        from google.colab import files
        files.upload()
        if Path("dim_item_itec.csv.gz").exists():
            _ungz("dim_item_itec.csv.gz", "dim_item_itec.csv")
    if not Path("finetune.py").exists():
        print("เลือกไฟล์  itec_finetune_scripts.zip")
        from google.colab import files
        files.upload()
        zipfile.ZipFile("itec_finetune_scripts.zip").extractall(".")
else:
    SCRIPTS = Path("/kaggle/working"); os.chdir(SCRIPTS)
    src = [p for p in Path("/kaggle/input").rglob("*") if p.is_file()]
    print("เจอใน /kaggle/input:", [p.name for p in src][:20])
    for p in src:
        if p.suffix == ".zip":
            zipfile.ZipFile(p).extractall(".")
        elif p.name.endswith(".csv.gz"):
            _ungz(p, p.name[:-3])
        elif not Path(p.name).exists():
            shutil.copy(p, p.name)

DATA = next((p for p in (SCRIPTS / "_data" / "dim_item_itec.csv",
                         SCRIPTS / "dim_item_itec.csv") if p.exists()), None)
assert DATA, f"ไม่พบ dim_item_itec.csv ใน {SCRIPTS}"
sys.path.insert(0, str(SCRIPTS))

import torch
print(f"env={ENV} · DATA={DATA.name}")
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "ไม่มี — fine-tune บน CPU จะช้ามาก ควรใช้ Kaggle GPU")

## 2 · เลือกค่าที่จะลอง

| | ค่า | ทำไม |
|---|---|---|
| `MODEL` | `mini` · `e5` · `bge` | **เริ่ม `mini` (118M)** พิสูจน์ว่ากระบวนการทำงานก่อน |
| `LABEL` | `Item_Type` · `Type_Platform` | **`Item_Type` มี ~20 หมวด ตัวอย่างต่อหมวดเยอะกว่า 12 เท่า** เห็นผลชัดกว่า |
| `N` | จำนวนแถว | 20,000 พอบอกทิศทาง · อย่ารันเต็มตั้งแต่รอบแรก |
| `EPOCHS` | 1 | เกิน 1 เสี่ยงจำข้อมูลแทนเรียน pattern |

In [ ]:
MODEL  = "mini"          # mini | e5 | bge
LABEL  = "Item_Type"     # Item_Type | Type_Platform | Item_Platform
N      = 20000
EPOCHS = 1
BATCH  = 32              # ต้องใหญ่พอให้มีหลายตัวอย่างต่อหมวดใน batch
MINPC  = 4               # ตัดหมวดที่มีน้อยกว่านี้ - triplet loss ต้องมีอย่างน้อย 2

print(f"{MODEL} · label={LABEL} · n={N:,} · {EPOCHS} epoch · batch {BATCH}")

## 3 · รัน

In [ ]:
import subprocess, sys
cmd = [sys.executable, "finetune.py", "--csv", str(DATA),
       "--model", MODEL, "--label", LABEL, "-n", str(N),
       "--epochs", str(EPOCHS), "--batch", str(BATCH),
       "--min-per-class", str(MINPC)]
print(" ".join(cmd), "\n")
subprocess.run(cmd)

## 4 · อ่านผล

| `f1_macro` ต่าง | ตัดสิน |
|---|---|
| **> +0.01** | คุ้ม · ใช้ backbone ที่ปรับแล้ว |
| −0.01 ถึง +0.01 | ไม่คุ้ม · ได้ของที่ต้องดูแลเพิ่มโดยไม่ได้อะไร |
| **< −0.01** | **แย่ลง** — label ยัง noisy เกินไป กลับไปทำ label สะอาดก่อน |

**ดู `f1_macro` ไม่ใช่ `accuracy`** — หมวดเบ้มาก accuracy บอกอะไรไม่ได้

In [ ]:
import pandas as pd
from pathlib import Path
p = Path("_out/finetune_result.csv")
display(pd.read_csv(p, encoding="utf-8-sig")) if p.exists() else print("ยังไม่มีผล")

## 5 · ถ้าคุ้ม — เอา backbone ไปใช้ต่อ

แก้ Cell 8 ในโน้ตบุ๊กหลัก (`../category/itec_recategorization.ipynb`)

```python
backbone = "_out/mini-Item_Type-finetuned"
```

แล้วรัน Cell 8–12 ใหม่ทั้งหมด

In [ ]:
from pathlib import Path
for p in sorted(Path("_out").glob("*finetuned")):
    print(p, "·", sum(f.stat().st_size for f in p.rglob("*") if f.is_file()) // 1024 // 1024, "MB")

## 6 · ดาวน์โหลดผล

In [ ]:
from pathlib import Path
import shutil
if ENV != "local":
    shutil.make_archive("finetune_out", "zip", "_out")
    print("_out/ ถูกบีบเป็น finetune_out.zip")
    if ENV == "colab":
        from google.colab import files
        files.download("finetune_out.zip")
    else:
        print("Kaggle: ดูที่แท็บ Output ทางขวา")
else:
    print("รันในเครื่อง ผลอยู่ใน _out/ แล้ว")

---

## ถ้า fine-tune ไม่ช่วย ทำอะไรต่อ

**อย่ารีบเพิ่ม epoch หรือเปลี่ยนโมเดลใหญ่ขึ้น** — มักไม่ใช่สาเหตุ

| ลองตามลำดับ | เหตุผล |
|---|---|
| 1. `LABEL = "Item_Type"` | หมวดน้อยลง ตัวอย่างต่อหมวดเยอะขึ้น |
| 2. `N = 50000` | 245 หมวดจาก 20k แถว = เฉลี่ยหมวดละ 80 ยังน้อย |
| 3. `MINPC = 10` | ตัดหมวดที่ตัวอย่างน้อยเกินไปออก |
| 4. **กลับไปทำ label สะอาด** | Cell 12 ในโน้ตบุ๊กหลัก — **คุ้มกว่าทุกข้อข้างบน** |

> 🔑 **fine-tune ดีขึ้นต่อเมื่อ label สะอาดก่อน** — ถ้ายัง noisy จะเรียนของผิดแม่นขึ้น
